# Pasos previos

## Imports y dependencias

In [ ]:
pip install scikit-learn pandas numpy gensim nltk

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.mixture import GaussianMixture
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import FunctionTransformer

import nltk
nltk.download('stopwords')
nltk.download('punkt')
import pandas as pd
import re
from nltk.tokenize import TweetTokenizer, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer


## Carga de archivo y preprocesamiento

In [ ]:
tknzr =  TweetTokenizer(preserve_case=False)
stemmer = SnowballStemmer('spanish')

nltk.download("stopwords", quiet=True)
stop_es = set(stopwords.words("spanish"))

def preprocesar(text): # Preprocesamiento hecho
  texto = text.strip()
  texto = tknzr.tokenize(texto)
  texto = [stemmer.stem(t) for t in texto if t not in stop_es]
  texto = ' '.join(texto)
  patron = r'[^\w\sáéíóúüñáàèìòùäëïöü]'
  texto = re.sub(patron, '', texto)
  texto = ' '.join(texto.split())
  return texto

In [ ]:
name = 'news_reducido.csv'

# Leer los datos en formato csv
data = pd.read_csv(name, on_bad_lines='skip')

# Nos quedamos con el texto (puedes quedarte con más información si quieres)
X = np.array([preprocesar(t) for t in data['text'].fillna('')])

## Datos utilizados

In [ ]:
semilla1=42
semilla2=640
semilla3=5300742

semilla = semilla2

#Agrupamiento
num_clusters=4



## Validación cruzada estratificada

In [ ]:
enc = OrdinalEncoder()
y = enc.fit_transform(np.reshape(data['category'], (-1,1))).reshape(-1) # DOCUMENTOS Y LE ASOCIO LA CATEGORIA

# Hacemos la partición train-test con Validacion cruzada estratificada
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=semilla)
skf.get_n_splits(X, y)

# Agrupamiento

## Definición de los modelos

### Representación binaria

In [ ]:
text_kmeans_binario = Pipeline([
    ('vect', CountVectorizer(binary=True)),
    ('clf', KMeans(n_clusters=num_clusters, random_state=semilla))
])

### Representación de frecuencias

In [ ]:
text_kmeans_tf = Pipeline([
    ('vect', CountVectorizer(binary=False)),
    ('clf', KMeans(n_clusters=num_clusters, random_state=semilla))
])

### Representación TF-IDF

In [ ]:
text_kmeans_tfidf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', KMeans(n_clusters=num_clusters, random_state=semilla))
])

### Algoritmo de mezcla gaussiana

In [ ]:
text_kmeans_tfidf_mezcla_gausiana = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('to_dense', FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
    ('clf', GaussianMixture(n_components=num_clusters, random_state=semilla, covariance_type='spherical'))
])

## Ejecución de los modelos

### Representación binaria

In [ ]:
text_kmeans = text_kmeans_binario

accuracies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])
    # Test
    labels = text_kmeans.predict(X[tst])
    # Calculo de metricas
    acc = np.mean(labels == y[tst])
    print("Fold " + str(i+1) + ": " + str(acc))
    accuracies[i] = acc

# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Representación de frecuencias

In [ ]:
text_kmeans = text_kmeans_tf

accuracies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])
    # Test
    labels = text_kmeans.predict(X[tst])
    # Calculo de metricas
    acc = np.mean(labels == y[tst])
    print("Fold " + str(i+1) + ": " + str(acc))
    accuracies[i] = acc

# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Representación TF-IDF

In [ ]:
text_kmeans = text_kmeans_tfidf

accuracies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])
    # Test
    labels = text_kmeans.predict(X[tst])
    # Calculo de metricas
    acc = np.mean(labels == y[tst])
    print("Fold " + str(i+1) + ": " + str(acc))
    accuracies[i] = acc

# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

### Algoritmo de mezcla gaussiana


In [ ]:
text_kmeans = text_kmeans_tfidf_mezcla_gausiana

accuracies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])
    # Test
    labels = text_kmeans.predict(X[tst])
    # Calculo de metricas
    acc = np.mean(labels == y[tst])
    print("Fold " + str(i+1) + ": " + str(acc))
    accuracies[i] = acc

# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interés, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

## Representación del mejor y el peor fold con el algoritmo T-SNE

### Representación: Binaria, Algoritmo: K-Means, Semilla: 640, Fold: 1

In [ ]:
# Este te saca las cosas con las fotos
text_kmeans = text_kmeans_binario

folds = list(skf.split(X,y))
tra, tst = folds[0]

# Entrenamiento
text_kmeans.fit(X[tra])
# Test
labels = text_kmeans.predict(X[tst])
# Calculo de metricas
acc = np.mean(labels == y[tst])
print(acc)

X_tst_transformed = text_kmeans[:-1].transform(X[tst])

tsne = TSNE(
    n_components=2,
    # Aseguramos que la perplejidad no sea mayor que el número de muestras
    perplexity=min(30, max(5, X_tst_transformed.shape[0] // 3)),
    learning_rate='auto',
    init='pca',
    random_state=42
)

tst_2d = tsne.fit_transform(X_tst_transformed)

plt.figure(figsize=(9, 6))
# Usamos 'labels' que son las predicciones que hiciste arriba
plt.scatter(tst_2d[:, 0], tst_2d[:, 1], c=labels, s=12, cmap='tab10', alpha=0.8)
plt.title(f"t-SNE fold {1} (Representación Binaria)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.colorbar(label="Cluster Predicho")
plt.tight_layout()
nombre_archivo = "./tsne_fold_1_clusters_tst.png"
plt.savefig(nombre_archivo, dpi=300, bbox_inches="tight")
print(f"Imagen guardada con éxito como: {nombre_archivo}")
plt.close()

### Representación: TF-IDF, Algoritmo: K-Means, Semilla: 640, Fold: 3

In [ ]:
# Este te saca las cosas con las fotos
text_kmeans = text_kmeans_tfidf

folds = list(skf.split(X,y))
# Cambiado a la tercera posición (índice 2)
tra, tst = folds[2]

# Entrenamiento
text_kmeans.fit(X[tra])
# Test
labels = text_kmeans.predict(X[tst])
# Calculo de metricas
acc = np.mean(labels == y[tst])
print(acc)

X_tst_transformed = text_kmeans[:-1].transform(X[tst])

tsne = TSNE(
    n_components=2,
    # Aseguramos que la perplejidad no sea mayor que el número de muestras
    perplexity=min(30, max(5, X_tst_transformed.shape[0] // 3)),
    learning_rate='auto',
    init='pca',
    random_state=42
)

tst_2d = tsne.fit_transform(X_tst_transformed)

plt.figure(figsize=(9, 6))
# Usamos 'labels' que son las predicciones que hiciste arriba
plt.scatter(tst_2d[:, 0], tst_2d[:, 1], c=labels, s=12, cmap='tab10', alpha=0.8)
# Actualizado el título y el nombre del archivo al Fold 3
plt.title(f"t-SNE fold {3} (Representación TF-IDF)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.colorbar(label="Cluster Predicho")
plt.tight_layout()
nombre_archivo = "./tsne_fold_3_clusters_tst.png"
plt.savefig(nombre_archivo, dpi=300, bbox_inches="tight")
print(f"Imagen guardada con éxito como: {nombre_archivo}")
plt.close()